In [125]:
import pandas as pd
df = pd.read_csv("Ecommerce_transactions.csv")
df

,Transaction_ID,User_Name,Age,Country,Product_Category,Purchase_Amount,Payment_Method,Transaction_Date
0,1,Sophia Rodriguez,62,Mexico,Books,743.47,Debit Card,2024-01-01
1,2,Sophia Rodriguez,41,UK,Grocery,373.13,Net Banking,2024-01-01
2,3,Ava Hall,26,Mexico,Electronics,987.28,UPI,2024-01-01
3,4,James White,50,Canada,Clothing,533.45,Net Banking,2024-01-01
4,5,James Lewis,38,Japan,Grocery,180.92,Cash on Delivery,2024-01-01
...,...,...,...,...,...,...,...,...
25034,25035,Olivia Walker,21,UK,Toys,490.81,Debit Card,2024-12-31
25035,25036,Ava Rodriguez,59,Germany,Electronics,277.52,UPI,2024-12-31
25036,25037,Olivia White,49,India,Electronics,854.01,Cash on Delivery,2024-12-31
25037,25038,Noah Hall,48,Australia,Clothing,895.05,PayPal,2024-12-31


In [126]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25039 entries, 0 to 25038
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction_ID    25039 non-null  int64  
 1   User_Name         25039 non-null  object 
 2   Age               25039 non-null  int64  
 3   Country           25039 non-null  object 
 4   Product_Category  25039 non-null  object 
 5   Purchase_Amount   25039 non-null  float64
 6   Payment_Method    25039 non-null  object 
 7   Transaction_Date  25039 non-null  object 
dtypes: float64(1), int64(2), object(5)
memory usage: 1.5+ MB


In [127]:
df.columns

Index(['Transaction_ID', 'User_Name', 'Age', 'Country', 'Product_Category',
       'Purchase_Amount', 'Payment_Method', 'Transaction_Date'],
      dtype='object')

In [128]:
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'], format='%Y-%m-%d')

In [129]:
df['Transaction_Date'].dtype

dtype('<M8[ns]')

In [130]:
df

,Transaction_ID,User_Name,Age,Country,Product_Category,Purchase_Amount,Payment_Method,Transaction_Date
0,1,Sophia Rodriguez,62,Mexico,Books,743.47,Debit Card,2024-01-01
1,2,Sophia Rodriguez,41,UK,Grocery,373.13,Net Banking,2024-01-01
2,3,Ava Hall,26,Mexico,Electronics,987.28,UPI,2024-01-01
3,4,James White,50,Canada,Clothing,533.45,Net Banking,2024-01-01
4,5,James Lewis,38,Japan,Grocery,180.92,Cash on Delivery,2024-01-01
...,...,...,...,...,...,...,...,...
25034,25035,Olivia Walker,21,UK,Toys,490.81,Debit Card,2024-12-31
25035,25036,Ava Rodriguez,59,Germany,Electronics,277.52,UPI,2024-12-31
25036,25037,Olivia White,49,India,Electronics,854.01,Cash on Delivery,2024-12-31
25037,25038,Noah Hall,48,Australia,Clothing,895.05,PayPal,2024-12-31


In [131]:
fee_map = {
    'UPI': 0.00,
    'Credit Card': 0.025,
    'PayPal': 0.035,
    'Cash on Delivery': 0.05,
    'Debit Card': 0.015,
    'Net Banking': 0.01
}

delay_map = {
    'UPI': 0,
    'Credit Card': 1,
    'PayPal': 3,
    'Cash on Delivery': 7,
    'Debit Card': 1,
    'Net Banking': 1
}


In [132]:
df['Fee_Rate'] = df['Payment_Method'].map(fee_map)
df['Delay_Days'] = df['Payment_Method'].map(delay_map)

In [133]:
df['Net_Amount'] = df['Purchase_Amount'] * (1 - df['Fee_Rate'])

In [134]:
df['Bank_Arrival_Date'] = df['Transaction_Date'] + pd.to_timedelta(df['Delay_Days'], unit='D')

In [135]:
df[['Payment_Method','Purchase_Amount','Fee_Rate',
    'Net_Amount','Delay_Days','Transaction_Date','Bank_Arrival_Date']].head(10)

,Payment_Method,Purchase_Amount,Fee_Rate,Net_Amount,Delay_Days,Transaction_Date,Bank_Arrival_Date
0,Debit Card,743.47,0.015,732.31795,1,2024-01-01,2024-01-02
1,Net Banking,373.13,0.010,369.39870,1,2024-01-01,2024-01-02
2,UPI,987.28,0.000,987.28000,0,2024-01-01,2024-01-01
3,Net Banking,533.45,0.010,528.11550,1,2024-01-01,2024-01-02
4,Cash on Delivery,180.92,0.050,171.87400,7,2024-01-01,2024-01-08
5,PayPal,904.99,0.035,873.31535,3,2024-01-01,2024-01-04
6,UPI,737.66,0.000,737.66000,0,2024-01-01,2024-01-01
7,UPI,767.89,0.000,767.89000,0,2024-01-01,2024-01-01
8,Debit Card,587.05,0.015,578.24425,1,2024-01-01,2024-01-02
9,Debit Card,741.82,0.015,730.69270,1,2024-01-01,2024-01-02


In [136]:
df[['Fee_Rate','Delay_Days']].isna().sum()

,0
Fee_Rate,0
Delay_Days,0


In [137]:
df['Arrival_Month'] = df['Bank_Arrival_Date'].dt.to_period('M')

In [138]:
monthly_cashflow = (
    df.groupby('Arrival_Month')
      .agg(
          Gross_Amount=('Purchase_Amount', 'sum'),
          Net_Amount=('Net_Amount', 'sum'),
          Total_Transactions=('Transaction_ID', 'count')
      )
      .reset_index()
)

monthly_cashflow['Gross_Amount'] = monthly_cashflow['Gross_Amount'].round(0)
monthly_cashflow['Net_Amount'] = monthly_cashflow['Net_Amount'].round(0)



monthly_cashflow


,Arrival_Month,Gross_Amount,Net_Amount,Total_Transactions
0,2024-01,1016165.0,995221.0,2026
1,2024-02,1021476.0,998115.0,2020
2,2024-03,1014188.0,991459.0,2041
3,2024-04,1044692.0,1020492.0,2060
4,2024-05,1063045.0,1039226.0,2132
5,2024-06,1024124.0,1000528.0,2005
6,2024-07,1079079.0,1054366.0,2126
7,2024-08,1065746.0,1042172.0,2088
8,2024-09,1040674.0,1017391.0,2046
9,2024-10,1067647.0,1043677.0,2115


In [139]:
df.to_csv("cashflow_transactions.csv", index=False)

In [140]:
monthly_cashflow.to_csv("monthly_cashflow.csv", index=False)